# ver17. Full Multi-Foundation Alzheimer MRI Pipeline

이 노트북은 최종 발표용으로 정리한 full pipeline입니다.

목표:

1. **SAM segmentation foundation model**로 MRI brain foreground mask/crop 생성
2. **BiomedCLIP adapter classifier**로 환자 단위 정상/비정상 screening 확률 계산
3. **LLM**으로 segmentation 결과와 classifier 결과를 통합한 한국어 report 생성

중요한 설계 원칙:

- SAM mask는 알츠하이머 병변 위치가 아니라 brain foreground 보조 정보입니다.
- 현재 검증된 classifier는 원본 BiomedCLIP feature 기반 adapter probe입니다.
- SAM crop을 곧바로 classifier 입력으로 쓰면 학습 분포가 달라지므로, crop 기반 재분류는 별도 실험으로 분리합니다.
- LLM은 의학적 진단을 생성하지 않고, 이미 계산된 값을 사람이 읽기 쉬운 report로 정리합니다.


## 1. 환경 점검


In [ ]:
import os
import sys
import re
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import requests

from IPython.display import Markdown, display
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python executable : {sys.executable}")
print(f"Python version    : {sys.version}")
print(f"torch version     : {torch.__version__}")
print(f"torch.version.cuda: {torch.version.cuda}")
print(f"cuda available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name          : {torch.cuda.get_device_name(0)}")
else:
    print("GPU name          : CUDA GPU not available")
print(f"device            : {device}")


## 2. 경로 및 실행 옵션

이 버전은 최종 연결용이므로 segmentation과 LLM을 조용히 fallback하지 않습니다.

LLM 연결 방법:

- LM Studio: 서버를 켠 뒤 보통 `http://localhost:1234/v1`
- Ollama OpenAI-compatible: Ollama를 켠 뒤 보통 `http://localhost:11434/v1`
- OpenAI API: `LLM_API_BASE="https://api.openai.com/v1"`, `LLM_API_KEY` 설정


In [ ]:
PROJECT_DIR = Path(r"C:\Users\user\alzheimer")
DATA_ROOT = Path(r"C:\Users\user\Desktop\alzheimer_dataset\Data")

FOUNDATION_DIR = PROJECT_DIR / "patient_level_stage1_foundation"
CACHE_PATH = FOUNDATION_DIR / "cache" / "non_vs_demented_biomedclip_all_original_features.pt"
MANIFEST_PATH = FOUNDATION_DIR / "non_vs_demented_biomedclip_all_images_manifest.csv"
PATIENT_TABLE_PATH = FOUNDATION_DIR / "non_vs_demented_biomedclip_patient_table.csv"
FOLD_RESULTS_PATH = FOUNDATION_DIR / "non_vs_demented_biomedclip_fold_results.csv"
CHECKPOINT_DIR = FOUNDATION_DIR / "checkpoints"

OUTPUT_DIR = PROJECT_DIR / "full_multifoundation_pipeline"
ASSET_DIR = OUTPUT_DIR / "demo_assets"
REPORT_DIR = OUTPUT_DIR / "llm_reports"
PROMPT_DIR = OUTPUT_DIR / "llm_prompts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ASSET_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
PROMPT_DIR.mkdir(parents=True, exist_ok=True)

# 실제 SAM 연결
SAM_MODEL_TYPE = "vit_b"
SAM_CHECKPOINT = Path(r"C:\Users\user\alzheimer\models\sam_vit_b_01ec64.pth")
REQUIRE_REAL_SAM = True

# 실제 LLM 연결
# 현재 기본값은 LM Studio입니다. Ollama를 쓰면 "http://localhost:11434/v1"로 바꾸세요.
LLM_API_BASE = os.environ.get("LLM_API_BASE", "http://localhost:1234/v1")
LLM_API_KEY = os.environ.get("LLM_API_KEY", "not-needed")
LLM_MODEL = os.environ.get("LLM_MODEL", "")
REQUIRE_REAL_LLM = True
LLM_TIMEOUT_SECONDS = 120

required_paths = [
    DATA_ROOT,
    FOUNDATION_DIR,
    CACHE_PATH,
    MANIFEST_PATH,
    PATIENT_TABLE_PATH,
    FOLD_RESULTS_PATH,
    CHECKPOINT_DIR,
    SAM_CHECKPOINT,
]
for path in required_paths:
    assert Path(path).exists(), f"필수 경로가 없습니다: {path}"

print(f"DATA_ROOT      : {DATA_ROOT}")
print(f"FOUNDATION_DIR : {FOUNDATION_DIR}")
print(f"OUTPUT_DIR     : {OUTPUT_DIR}")
print(f"SAM checkpoint : {SAM_CHECKPOINT}")
print(f"LLM_API_BASE   : {LLM_API_BASE}")
print(f"LLM_MODEL      : {LLM_MODEL if LLM_MODEL else '(models endpoint에서 자동 선택 시도)'}")


## 3. SAM segmentation model 실제 로드


In [ ]:
try:
    from segment_anything import SamPredictor, sam_model_registry
except Exception as exc:
    raise RuntimeError(
        "segment_anything 패키지가 필요합니다. "
        "현재 alzheimer 환경에서 `pip install segment-anything opencv-python requests`를 실행하세요."
    ) from exc

assert SAM_CHECKPOINT.exists(), f"SAM checkpoint 없음: {SAM_CHECKPOINT}"

sam_model = sam_model_registry[SAM_MODEL_TYPE](checkpoint=str(SAM_CHECKPOINT))
sam_model = sam_model.to(device)
sam_model.eval()
sam_predictor = SamPredictor(sam_model)

print(f"SAM loaded: {SAM_MODEL_TYPE}")
print(f"SAM checkpoint: {SAM_CHECKPOINT}")
print(f"SAM device: {next(sam_model.parameters()).device}")
assert str(next(sam_model.parameters()).device).startswith(str(device)), "SAM이 지정 device에 올라가지 않았습니다."


## 4. BiomedCLIP adapter classifier 결과 복원

분류기는 기존 ver11에서 검증한 frozen BiomedCLIP feature + adapter probe를 사용합니다. 여기서는 재학습하지 않고 저장된 feature cache와 checkpoint로 환자 단위 out-of-fold 예측을 복원합니다.


In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
patient_table = pd.read_csv(PATIENT_TABLE_PATH)
fold_results = pd.read_csv(FOLD_RESULTS_PATH)
cached_features = torch.load(CACHE_PATH, map_location="cpu", weights_only=False)

adapter_fold_results = (
    fold_results[fold_results["experiment"] == "adapter_probe"]
    .sort_values("fold")
    .reset_index(drop=True)
)
assert len(adapter_fold_results) == 5, "adapter_probe 5-fold 결과가 필요합니다."

print(f"Images   : {len(manifest):,}")
print(f"Patients : {len(patient_table):,}")
print(f"Feature tensor shape: {tuple(cached_features['features'].shape)}")
display(adapter_fold_results)


In [ ]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
patient_indices = np.arange(len(patient_table))
patient_targets = patient_table["target"].to_numpy()

fold_assignment = {}
for fold_number, (_, outer_test_idx) in enumerate(
    outer_cv.split(patient_indices, patient_targets),
    start=1,
):
    for patient_id in patient_table.iloc[outer_test_idx]["patient_id"]:
        fold_assignment[patient_id] = fold_number

patient_table = patient_table.copy()
patient_table["fold"] = patient_table["patient_id"].map(fold_assignment)
assert patient_table["fold"].notna().all()

display(
    patient_table.groupby(["fold", "target"])
    .size()
    .unstack(fill_value=0)
)


In [ ]:
ADAPTER_HIDDEN_DIM = 128
ADAPTER_DROPOUT = 0.2
PREDICT_BATCH_SIZE = 4096
feature_dim = int(cached_features["features"].shape[1])


class AdapterProbe(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.norm = nn.LayerNorm(input_dim)
        self.adapter = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, input_dim),
        )
        self.classifier = nn.Linear(input_dim, 2)

    def forward(self, features):
        normalized = self.norm(features)
        adapted = normalized + self.adapter(normalized)
        return self.classifier(adapted)


def load_adapter_checkpoint(fold_number: int):
    checkpoint_path = CHECKPOINT_DIR / f"adapter_probe_fold{fold_number}.pt"
    assert checkpoint_path.exists(), f"checkpoint 없음: {checkpoint_path}"
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )
    model = AdapterProbe(
        input_dim=int(checkpoint["feature_dim"]),
        hidden_dim=ADAPTER_HIDDEN_DIM,
        dropout=ADAPTER_DROPOUT,
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(device)
    model.eval()
    return model, checkpoint


def aggregate_patient_predictions(patient_ids, labels, probabilities):
    df = pd.DataFrame({
        "patient_id": patient_ids,
        "target": labels,
        "image_probability": probabilities,
    })
    return (
        df.groupby("patient_id", as_index=False)
        .agg(
            target=("target", "first"),
            probability=("image_probability", "mean"),
            image_probability_std=("image_probability", "std"),
            image_count=("image_probability", "count"),
        )
    )


def predict_patients_with_fold_checkpoint(fold_number: int, patient_ids):
    patient_id_set = set(patient_ids)
    indices = [
        index
        for index, patient_id in enumerate(cached_features["patient_ids"])
        if patient_id in patient_id_set
    ]
    assert indices, f"fold {fold_number}에 해당하는 이미지 index가 없습니다."

    model, checkpoint = load_adapter_checkpoint(fold_number)
    features = cached_features["features"][indices].float()
    labels = cached_features["labels"][indices].long().numpy()
    batch_patient_ids = [cached_features["patient_ids"][i] for i in indices]

    probabilities = []
    with torch.inference_mode():
        for start in range(0, len(indices), PREDICT_BATCH_SIZE):
            end = min(start + PREDICT_BATCH_SIZE, len(indices))
            batch_features = features[start:end].to(device, non_blocking=True)
            logits = model(batch_features)
            batch_probs = logits.softmax(dim=1)[:, 1]
            probabilities.extend(batch_probs.detach().cpu().numpy().tolist())

    patient_pred = aggregate_patient_predictions(
        batch_patient_ids,
        labels,
        probabilities,
    )
    patient_pred["fold"] = fold_number
    patient_pred["threshold"] = float(checkpoint["selected_threshold"])
    patient_pred["pred_label"] = (
        patient_pred["probability"] >= patient_pred["threshold"]
    ).astype(int)
    return patient_pred


def calculate_oof_metrics(oof_df):
    y_true = oof_df["target"].to_numpy()
    y_prob = oof_df["probability"].to_numpy()
    y_pred = oof_df["pred_label"].to_numpy()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "patients": len(oof_df),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / max(tn + fp, 1),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "auroc": roc_auc_score(y_true, y_prob),
        "auprc": average_precision_score(y_true, y_prob),
    }


oof_parts = []
for fold_number in range(1, 6):
    fold_patient_ids = patient_table.loc[
        patient_table["fold"] == fold_number,
        "patient_id",
    ].tolist()
    oof_parts.append(
        predict_patients_with_fold_checkpoint(fold_number, fold_patient_ids)
    )

oof_predictions = pd.concat(oof_parts, ignore_index=True)
oof_predictions = oof_predictions.merge(
    patient_table[["patient_id", "class_name"]],
    on="patient_id",
    how="left",
)
oof_predictions = oof_predictions[
    [
        "patient_id",
        "class_name",
        "target",
        "fold",
        "image_count",
        "probability",
        "image_probability_std",
        "threshold",
        "pred_label",
    ]
].sort_values(["fold", "patient_id"])

oof_metrics = calculate_oof_metrics(oof_predictions)
oof_predictions.to_csv(
    OUTPUT_DIR / "adapter_probe_oof_patient_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)
(OUTPUT_DIR / "adapter_probe_oof_metrics.json").write_text(
    json.dumps(oof_metrics, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("[OOF metrics]")
for key, value in oof_metrics.items():
    if isinstance(value, float):
        print(f"{key:12s}: {value:.4f}")
    else:
        print(f"{key:12s}: {value}")
display(oof_predictions.head())


## 5. 대표 환자 선택


In [ ]:
def add_first_unique_case(case_rows, case_type, selected_rows, selected_ids):
    if case_rows.empty:
        return
    for _, row in case_rows.iterrows():
        patient_id = row["patient_id"]
        if patient_id in selected_ids:
            continue
        new_row = row.copy()
        new_row["case_type"] = case_type
        selected_rows.append(new_row)
        selected_ids.add(patient_id)
        return


def select_demo_cases(oof_df):
    work = oof_df.copy()
    work["threshold_distance"] = (
        work["probability"] - work["threshold"]
    ).abs()
    selected_rows = []
    selected_ids = set()

    add_first_unique_case(
        work[(work["target"] == 0) & (work["pred_label"] == 0)]
        .sort_values("probability", ascending=True),
        "정상으로 맞힌 낮은 위험도 사례",
        selected_rows,
        selected_ids,
    )
    add_first_unique_case(
        work[(work["target"] == 1) & (work["pred_label"] == 1)]
        .sort_values("probability", ascending=False),
        "비정상으로 맞힌 높은 위험도 사례",
        selected_rows,
        selected_ids,
    )
    add_first_unique_case(
        work.sort_values("threshold_distance", ascending=True),
        "threshold 근처의 경계 사례",
        selected_rows,
        selected_ids,
    )
    add_first_unique_case(
        work[work["target"] != work["pred_label"]]
        .sort_values("threshold_distance", ascending=True),
        "오분류 사례",
        selected_rows,
        selected_ids,
    )
    return pd.DataFrame(selected_rows).reset_index(drop=True)


def representative_image_path(patient_id):
    paths = sorted(
        manifest.loc[manifest["patient_id"] == patient_id, "image_path"]
        .astype(str)
        .tolist()
    )
    assert paths, f"이미지를 찾을 수 없습니다: {patient_id}"
    return paths[len(paths) // 2]


demo_cases = select_demo_cases(oof_predictions)
demo_cases["representative_image_path"] = demo_cases["patient_id"].apply(
    representative_image_path
)

display(
    demo_cases[
        [
            "case_type",
            "patient_id",
            "class_name",
            "target",
            "fold",
            "probability",
            "threshold",
            "pred_label",
            "representative_image_path",
        ]
    ]
)


## 6. 실제 SAM segmentation 실행

SAM에는 직접 의학 ROI를 지정하지 않습니다. 대신 MRI의 밝은 foreground 영역을 이용해 자동 box prompt를 만들고, SAM이 그 box 안의 주요 객체를 mask로 분리하게 합니다.

알츠하이머 MRI는 종양처럼 작은 병변을 찾는 문제가 아니기 때문에 mask가 넓게 나오는 것이 이상한 결과는 아닙니다. 이 단계의 목적은 hippocampus 같은 전문 해부학 ROI를 표시하는 것이 아니라, 검은 배경을 줄이고 전체 뇌 context를 시각화하는 것입니다.


In [ ]:
def read_rgb_image(image_path):
    return Image.open(image_path).convert("RGB")


def automatic_foreground_box(image, pad=10):
    gray = np.asarray(image.convert("L"))
    nonzero = gray[gray > 0]
    if len(nonzero) == 0:
        h, w = gray.shape
        return (0, 0, w - 1, h - 1)

    threshold = max(8, int(np.percentile(nonzero, 20) * 0.65))
    mask = gray > threshold
    ys, xs = np.where(mask)
    h, w = gray.shape
    if len(xs) == 0 or len(ys) == 0:
        return (0, 0, w - 1, h - 1)
    x1 = max(int(xs.min()) - pad, 0)
    y1 = max(int(ys.min()) - pad, 0)
    x2 = min(int(xs.max()) + pad, w - 1)
    y2 = min(int(ys.max()) + pad, h - 1)
    return (x1, y1, x2, y2)


def bbox_from_mask(mask, pad=8):
    ys, xs = np.where(mask)
    h, w = mask.shape
    if len(xs) == 0 or len(ys) == 0:
        return (0, 0, w - 1, h - 1)
    return (
        max(int(xs.min()) - pad, 0),
        max(int(ys.min()) - pad, 0),
        min(int(xs.max()) + pad, w - 1),
        min(int(ys.max()) + pad, h - 1),
    )


def keep_largest_component(mask):
    mask = mask.astype(bool)
    try:
        from scipy import ndimage as ndi

        labeled, num_labels = ndi.label(mask)
        if num_labels == 0:
            return mask
        sizes = np.bincount(labeled.ravel())
        sizes[0] = 0
        return labeled == int(sizes.argmax())
    except Exception:
        return mask


def run_sam_segmentation(image):
    image_np = np.asarray(image)
    box = automatic_foreground_box(image)
    sam_predictor.set_image(image_np)
    masks, scores, _ = sam_predictor.predict(
        box=np.asarray(box, dtype=np.float32),
        multimask_output=True,
    )
    best_index = int(np.argmax(scores))
    mask = keep_largest_component(masks[best_index].astype(bool))
    refined_bbox = bbox_from_mask(mask, pad=8)
    mask_ratio = float(mask.mean())
    h, w = mask.shape
    x1, y1, x2, y2 = refined_bbox
    bbox_area_ratio = float(((x2 - x1 + 1) * (y2 - y1 + 1)) / max(h * w, 1))
    if bbox_area_ratio >= 0.75:
        scope_note = "wide whole-brain foreground"
    elif bbox_area_ratio >= 0.45:
        scope_note = "moderate brain foreground"
    else:
        scope_note = "tight foreground"
    return {
        "mask": mask,
        "box_prompt": box,
        "bbox": refined_bbox,
        "sam_score": float(scores[best_index]),
        "mask_ratio": mask_ratio,
        "bbox_area_ratio": bbox_area_ratio,
        "segmentation_scope_note": scope_note,
    }


def save_sam_assets(image_path, patient_id):
    image = read_rgb_image(image_path)
    result = run_sam_segmentation(image)
    mask = result["mask"]
    bbox = result["bbox"]

    image_np = np.asarray(image).astype(np.float32)
    overlay_np = image_np.copy()
    color = np.asarray([255, 70, 40], dtype=np.float32)
    alpha = 0.35
    overlay_np[mask] = (1 - alpha) * overlay_np[mask] + alpha * color
    overlay = Image.fromarray(np.clip(overlay_np, 0, 255).astype(np.uint8))

    draw = ImageDraw.Draw(overlay)
    draw.rectangle(bbox, outline=(80, 255, 120), width=2)

    x1, y1, x2, y2 = bbox
    crop = image.crop((x1, y1, x2 + 1, y2 + 1))
    mask_img = Image.fromarray(mask.astype(np.uint8) * 255)

    safe_id = re.sub(r"[^A-Za-z0-9_]+", "_", patient_id)
    overlay_path = ASSET_DIR / f"{safe_id}_sam_overlay.png"
    crop_path = ASSET_DIR / f"{safe_id}_sam_crop.png"
    mask_path = ASSET_DIR / f"{safe_id}_sam_mask.png"

    overlay.save(overlay_path)
    crop.save(crop_path)
    mask_img.save(mask_path)

    return {
        "segmentation_backend": f"SAM {SAM_MODEL_TYPE}",
        "sam_score": result["sam_score"],
        "mask_ratio": result["mask_ratio"],
        "bbox_area_ratio": result["bbox_area_ratio"],
        "segmentation_scope_note": result["segmentation_scope_note"],
        "box_prompt": result["box_prompt"],
        "bbox": result["bbox"],
        "overlay_path": str(overlay_path),
        "crop_path": str(crop_path),
        "mask_path": str(mask_path),
    }


asset_rows = []
for _, row in demo_cases.iterrows():
    asset_rows.append(
        save_sam_assets(
            row["representative_image_path"],
            row["patient_id"],
        )
    )

asset_df = pd.DataFrame(asset_rows)
demo_cases_with_assets = pd.concat(
    [demo_cases.reset_index(drop=True), asset_df],
    axis=1,
)

display(
    demo_cases_with_assets[
        [
            "case_type",
            "patient_id",
            "probability",
            "threshold",
            "pred_label",
            "segmentation_backend",
            "sam_score",
            "mask_ratio",
            "bbox_area_ratio",
            "segmentation_scope_note",
            "overlay_path",
        ]
    ]
)


In [ ]:
fig, axes = plt.subplots(
    len(demo_cases_with_assets),
    2,
    figsize=(8, 4 * len(demo_cases_with_assets)),
)
if len(demo_cases_with_assets) == 1:
    axes = np.asarray([axes])

for row_index, (_, row) in enumerate(demo_cases_with_assets.iterrows()):
    original = Image.open(row["representative_image_path"]).convert("RGB")
    overlay = Image.open(row["overlay_path"]).convert("RGB")
    axes[row_index, 0].imshow(original)
    axes[row_index, 0].set_title(
        f"Original | {row['patient_id']} | {row['class_name']}"
    )
    axes[row_index, 0].axis("off")
    axes[row_index, 1].imshow(overlay)
    axes[row_index, 1].set_title(
        f"SAM foreground | prob={row['probability']:.3f}, th={row['threshold']:.3f}"
    )
    axes[row_index, 1].axis("off")

plt.tight_layout()
preview_path = ASSET_DIR / "sam_segmentation_classifier_preview.png"
plt.savefig(preview_path, dpi=160, bbox_inches="tight")
plt.show()

print(f"Preview saved: {preview_path}")


## 7. 실제 LLM endpoint 연결 확인


In [ ]:
def normalize_api_base(base_url):
    return base_url.rstrip("/")


def list_llm_models():
    url = normalize_api_base(LLM_API_BASE) + "/models"
    headers = {"Authorization": f"Bearer {LLM_API_KEY}"}
    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()
    data = response.json()
    model_ids = [item["id"] for item in data.get("data", []) if "id" in item]
    return model_ids, data


try:
    available_models, raw_model_response = list_llm_models()
except Exception as exc:
    raise RuntimeError(
        "LLM endpoint에 연결할 수 없습니다.\n"
        f"현재 LLM_API_BASE={LLM_API_BASE}\n\n"
        "해결 방법:\n"
        "1. LM Studio를 쓰면 Local Server를 켜고 LLM_API_BASE를 http://localhost:1234/v1 로 둡니다.\n"
        "2. Ollama를 쓰면 Ollama를 실행하고 LLM_API_BASE를 http://localhost:11434/v1 로 바꿉니다.\n"
        "3. OpenAI API를 쓰면 LLM_API_BASE=https://api.openai.com/v1, LLM_API_KEY를 설정합니다.\n"
    ) from exc

if not LLM_MODEL:
    assert available_models, "models endpoint는 열렸지만 사용 가능한 model id가 없습니다."
    LLM_MODEL = available_models[0]

print("LLM endpoint connected.")
print(f"LLM_API_BASE: {LLM_API_BASE}")
print(f"LLM_MODEL   : {LLM_MODEL}")
print("Available models:", available_models[:10])


## 8. LLM prompt와 report 생성


In [ ]:
SYSTEM_PROMPT = """
너는 의료 진단을 확정하는 의사가 아니다.
너의 역할은 이미 계산된 MRI screening model 결과와 SAM segmentation 결과를 사람이 읽기 쉬운 한국어 보고서로 정리하는 것이다.
모델이 제공하지 않은 의학적 소견을 새로 만들어내지 마라.
SAM mask는 알츠하이머 병변 위치가 아니라 brain foreground 보조 시각화라고 설명하라.
최종 문장에는 이 결과가 진단이 아니라 연구용 screening 결과임을 분명히 적어라.
""".strip()


def model_decision_text(pred_label):
    return "비정상 의심" if int(pred_label) == 1 else "정상 가능성"


def confidence_band(probability, threshold):
    distance = abs(float(probability) - float(threshold))
    if distance < 0.05:
        return "threshold에 매우 가까운 경계 결과"
    if distance < 0.15:
        return "중간 정도의 확신"
    return "상대적으로 명확한 결과"


def build_case_payload(row):
    return {
        "patient_id": str(row["patient_id"]),
        "case_type": str(row["case_type"]),
        "experimental_ground_truth_class": str(row["class_name"]),
        "segmentation_model": str(row["segmentation_backend"]),
        "sam_score": round(float(row["sam_score"]), 4),
        "mask_ratio": round(float(row["mask_ratio"]), 4),
        "bbox_area_ratio": round(float(row["bbox_area_ratio"]), 4),
        "segmentation_scope_note": str(row["segmentation_scope_note"]),
        "classifier": "BiomedCLIP frozen image encoder + adapter probe",
        "classifier_input": "original MRI feature cache, patient-level aggregation",
        "task": "Stage 1 binary screening: NonDemented vs Demented",
        "probability_demented": round(float(row["probability"]), 4),
        "threshold": round(float(row["threshold"]), 4),
        "prediction": model_decision_text(row["pred_label"]),
        "confidence_band": confidence_band(row["probability"], row["threshold"]),
        "mask_interpretation": (
            "SAM 기반 brain foreground 보조 mask이며, "
            "알츠하이머 병변 위치나 전문의 annotation이 아님"
        ),
        "overlay_path": str(row["overlay_path"]),
        "crop_path": str(row["crop_path"]),
    }


def build_llm_prompt(payload):
    return f"""
아래 JSON은 Alzheimer MRI multi-foundation screening pipeline의 한 환자 예시 결과다.
이 정보를 바탕으로 한국어 보고서를 작성하라.

보고서 형식:
1. 입력 및 파이프라인 요약
2. SAM segmentation 결과
3. BiomedCLIP adapter classifier 판단
4. 통합 해석
5. 해석상 주의점

제약:
- 진단, 확진, 병변 검출이라고 표현하지 말 것
- SAM mask를 병변 위치로 해석하지 말 것
- segmentation_scope_note가 wide whole-brain foreground라면 넓은 전체 뇌 context mask라고 설명할 것
- ground truth class는 실험 검증용 metadata로만 다룰 것
- MRI에서 관찰되지 않은 의학적 소견을 생성하지 말 것

JSON:
{json.dumps(payload, ensure_ascii=False, indent=2)}
""".strip()


def call_llm(prompt):
    url = normalize_api_base(LLM_API_BASE) + "/chat/completions"
    headers = {
        "Authorization": f"Bearer {LLM_API_KEY}",
        "Content-Type": "application/json",
    }
    body = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        "temperature": 0.2,
    }
    response = requests.post(
        url,
        headers=headers,
        json=body,
        timeout=LLM_TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"]


In [ ]:
report_records = []

for _, row in demo_cases_with_assets.iterrows():
    payload = build_case_payload(row)
    prompt = build_llm_prompt(payload)
    report = call_llm(prompt)

    safe_id = re.sub(r"[^A-Za-z0-9_]+", "_", payload["patient_id"])
    prompt_path = PROMPT_DIR / f"{safe_id}_prompt.md"
    report_path = REPORT_DIR / f"{safe_id}_llm_report.md"
    prompt_path.write_text(prompt, encoding="utf-8")
    report_path.write_text(report, encoding="utf-8")

    report_records.append({
        **payload,
        "llm_api_base": LLM_API_BASE,
        "llm_model": LLM_MODEL,
        "prompt_path": str(prompt_path),
        "report_path": str(report_path),
    })

report_df = pd.DataFrame(report_records)
report_index_path = OUTPUT_DIR / "full_pipeline_report_index.csv"
report_df.to_csv(report_index_path, index=False, encoding="utf-8-sig")

display(
    report_df[
        [
            "patient_id",
            "case_type",
            "segmentation_model",
            "classifier",
            "prediction",
            "probability_demented",
            "threshold",
            "llm_model",
            "report_path",
        ]
    ]
)
print(f"Report index saved: {report_index_path}")


In [ ]:
for _, row in report_df.iterrows():
    display(Markdown(Path(row["report_path"]).read_text(encoding="utf-8")))


## 9. 최종 발표 문장


In [ ]:
final_text = f"""
# Final Multi-Foundation Pipeline

본 프로젝트는 Alzheimer MRI 이미지를 대상으로 세 가지 foundation-model 기반 모듈을 연결했다.

1. SAM segmentation foundation model
- MRI slice에서 자동 box prompt를 생성하고, SAM ViT-B로 brain foreground mask를 만든다.
- 이 mask는 알츠하이머 병변 위치가 아니라 입력 영역 정제와 시각화 보조를 위한 결과다.
- 알츠하이머는 작은 국소 병변 검출 문제가 아니므로, SAM 결과는 좁은 lesion ROI가 아니라 넓은 whole-brain context로 해석한다.

2. BiomedCLIP adapter classifier
- frozen BiomedCLIP image encoder에서 추출한 feature에 adapter probe를 붙여 정상/비정상 screening을 수행한다.
- 환자 단위 5-fold 검증으로 data leakage를 줄였다.
- 복원된 OOF 성능은 sensitivity={oof_metrics['sensitivity']:.3f}, specificity={oof_metrics['specificity']:.3f},
  F1={oof_metrics['f1']:.3f}, AUROC={oof_metrics['auroc']:.3f}, AUPRC={oof_metrics['auprc']:.3f}이다.

3. LLM report module
- SAM 결과, BiomedCLIP 확률, threshold, 판단 결과를 structured prompt로 전달한다.
- LLM은 진단을 새로 생성하지 않고, 연구용 screening 결과를 사람이 읽기 쉬운 한국어 보고서로 변환한다.

따라서 최종 구조는 MRI -> SAM segmentation -> BiomedCLIP adapter classification -> LLM report로 이어지는
multi-foundation Alzheimer MRI screening assistant이다.
""".strip()

final_text_path = OUTPUT_DIR / "final_presentation_text.md"
final_text_path.write_text(final_text, encoding="utf-8")
display(Markdown(final_text))
print(f"Saved: {final_text_path}")
